# Model Serving & Latency Optimization

Production serving is where algorithmic ML meets systems engineering. This note covers REST vs gRPC, dynamic batching simulation, quantization, and why p99 latency matters more than mean latency.

## What Interviewers Test
- Dynamic batching: why it improves throughput and what it trades for latency
- Quantization: fp16/int8 size-speed tradeoffs
- p50/p95/p99 percentile latency — why tails matter
- REST vs gRPC tradeoffs
- When to use horizontal scaling vs model optimization

## REST vs gRPC

| | REST/HTTP | gRPC/HTTP2 |
|---|---|---|
| **Protocol** | JSON over HTTP/1.1 | Protobuf over HTTP/2 |
| **Payload size** | Large (JSON text) | Small (binary) |
| **Latency** | Higher | Lower (multiplexing) |
| **Streaming** | Limited | Native |
| **Tooling** | Universal (curl, browser) | Needs protobuf toolchain |
| **Use when** | External APIs, debugging | Internal microservices, high-throughput |

> 💡 **Interview Tip:** For ML serving at FAANG, gRPC is standard for internal services (model servers called by application servers). REST is used for external APIs where tooling simplicity matters.


In [ ]:
import numpy as np
import time
import queue, threading
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)

# --- Dynamic batching simulation ---
def simulate_static_batching(n_requests=200, model_latency_ms=10, qps=50):
    """Static batching: process each request individually."""
    arrival_interval = 1.0 / qps  # seconds between requests
    latencies = []
    for i in range(n_requests):
        t_arrive = i * arrival_interval
        t_serve  = t_arrive + model_latency_ms / 1000
        latencies.append(model_latency_ms)
    throughput = n_requests / (n_requests * arrival_interval + model_latency_ms/1000)
    return latencies, throughput

def simulate_dynamic_batching(n_requests=200, model_latency_ms=10, qps=50,
                               max_batch=16, max_wait_ms=5):
    """Dynamic batching: collect requests up to max_batch or max_wait_ms."""
    arrival_interval = 1.0 / qps
    latencies = []
    i = 0
    t = 0.0
    while i < n_requests:
        # Collect a batch
        batch_size = min(max_batch, n_requests - i)
        batch_collect_time = min(arrival_interval * batch_size, max_wait_ms / 1000)
        # Process batch (amortize model_latency_ms across batch)
        process_time = model_latency_ms / 1000  # batch size doesn't change compute much
        for j in range(batch_size):
            wait_time = batch_collect_time - j * arrival_interval
            latencies.append(max(0, wait_time) * 1000 + model_latency_ms)
        t += batch_collect_time + process_time
        i += batch_size
    throughput = n_requests / (n_requests * arrival_interval)
    return latencies, throughput

stat_lat, stat_tp = simulate_static_batching()
dyn_lat,  dyn_tp  = simulate_dynamic_batching()

print("=== Static vs Dynamic Batching ===")
print(f"Static:  throughput={stat_tp:.1f} req/s, p50={np.percentile(stat_lat,50):.1f}ms, p99={np.percentile(stat_lat,99):.1f}ms")
print(f"Dynamic: throughput={dyn_tp:.1f} req/s,  p50={np.percentile(dyn_lat,50):.1f}ms, p99={np.percentile(dyn_lat,99):.1f}ms")


In [ ]:
import torch
import torch.nn as nn

# --- PyTorch dynamic quantization ---
class MediumModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, 512), nn.ReLU(),
            nn.Linear(512, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.net(x)

model = MediumModel()

# Dynamic quantization (int8)
model_quant = torch.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8
)

# Compare model size
def model_size_mb(m):
    total = sum(p.numel() * p.element_size() for p in m.parameters())
    # Quantized params are int8 (1 byte)
    return total / 1e6

x = torch.randn(1, 256)

# Speed benchmark
def time_inference(m, x, n=100):
    m.eval()
    with torch.no_grad():
        t0 = time.time()
        for _ in range(n):
            m(x)
        return (time.time() - t0) / n * 1000  # ms per inference

fp32_ms   = time_inference(model, x)
quant_ms  = time_inference(model_quant, x)

print("=== Quantization ===")
print(f"FP32 model size: ~{sum(p.numel() for p in model.parameters()) * 4 / 1e6:.2f} MB")
print(f"INT8 model size: ~{sum(p.numel() for p in model.parameters()) * 1 / 1e6:.2f} MB (4x reduction)")
print(f"FP32 latency: {fp32_ms:.3f}ms per inference")
print(f"INT8 latency: {quant_ms:.3f}ms per inference")


In [ ]:
# --- Percentile latency analysis ---
# Simulate a realistic latency distribution (long-tail)
latency_samples = np.concatenate([
    np.random.lognormal(mean=2.5, sigma=0.4, size=950),  # normal requests
    np.random.lognormal(mean=4.0, sigma=0.5, size=50),   # slow tail (GC, cold cache)
])

print("=== Latency Percentiles ===")
for p in [50, 75, 90, 95, 99, 99.9]:
    print(f"  p{p}: {np.percentile(latency_samples, p):.1f}ms")

print(f"\nMean: {latency_samples.mean():.1f}ms (misleadingly low)")
print(f"Max:  {latency_samples.max():.1f}ms")
print()
print("Why p99 matters: 1% of requests = 1 in 100 users sees slow response.")
print("At 10K QPS, that's 100 users/second experiencing latency issues.")
print("SLAs are defined on percentiles, not means.")


## Common Interview Questions

**Q: What is dynamic batching and what does it trade off?**
Dynamic batching collects requests arriving within a short window (5–10ms) and processes them as a single batch. This improves GPU utilization and throughput (amortizing fixed batch overhead), at the cost of slightly increased latency for individual requests. The tradeoff: higher throughput vs lower per-request latency.

**Q: Why do SLAs use p99 instead of mean latency?**
Mean is pulled down by many fast requests and masks the slow tail. At 1K QPS with p99 = 200ms, one user per second experiences a slow response — which feels like a broken product. User satisfaction is dominated by the worst-case experience, not the average. Companies define SLAs at p95 or p99 because this better reflects user experience.

**Q: What is quantization and what are the quality tradeoffs?**
Quantization reduces numerical precision of model weights and activations: fp32 → fp16 (2× memory) → int8 (4× memory vs fp32). INT8 quantization typically degrades accuracy by <0.5% for well-trained models but significantly reduces memory footprint and increases throughput on hardware with int8 SIMD support. INT4 is aggressive and may require quantization-aware training.

**Q: When should you scale horizontally vs optimize the model?**
Start with model optimization (quantization, distillation, batching) to reduce per-request cost. When utilization is high and optimization is exhausted, scale horizontally (more replicas). If the bottleneck is memory-bound (large model, small batch), vertical scaling (larger GPU) may be more efficient than horizontal.

## Key Takeaways
- gRPC (internal) vs REST (external): binary protocol, lower latency vs universal tooling
- Dynamic batching: improve throughput by batching concurrent requests with a short wait window
- INT8 quantization: 4× memory reduction, 2–4× speedup, <0.5% accuracy loss for most models
- p99 latency is the SLA metric — mean is misleading; tail latency affects real users
- Quantization → distillation → batching → horizontal scaling: optimize in order of impact
- Always profile before optimizing; bottlenecks may be in data preprocessing, not model inference